In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.__version__

'2.14.0+cu132'

In [3]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, bias=True):
        super().__init__()
        self.out_channels = out_channels

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size, stride, padding, bias=bias)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)

        return x

class DownSample(nn.Module):
    def __init__(self, kernel_size, stride, padding):
        super().__init__()
        self.maxpool = nn.MaxPool2d(kernel_size, stride, padding)

    def forward(self, x):
        x = self.maxpool(x)

        return x

class UpSample(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding, bias=True):
        super().__init__()
        self.conv = nn.ConvTranspose2d(in_channels, out_channels, kernel_size, stride, padding, bias=bias)

    def forward(self, x):
        x = self.conv(x)

        return x

class UNet(nn.Module):
    def __init__(self, in_channels, out_channels, depth=4):
        super().__init__()
        self.depth = depth
        self.encoder_channels = [in_channels] + [64*(2**i) for i in range(depth)]
        self.decoder_channels = [64*(2**i) for i in range(depth+1)][::-1]

        self.encoder = nn.ModuleList(
            [nn.ModuleList(
                [DoubleConv(in_channels=self.encoder_channels[i], out_channels=self.encoder_channels[i+1], kernel_size=3, stride=1, padding=0),
                DownSample(kernel_size=2, stride=2, padding=0)]
            ) for i in range(depth)]
        )

        self.bottleneck = DoubleConv(in_channels=self.encoder_channels[-1], out_channels=self.decoder_channels[0], kernel_size=3, stride=1, padding=0)

        self.decoder = nn.ModuleList(
            [nn.ModuleList(
                [UpSample(in_channels=self.decoder_channels[i], out_channels=self.decoder_channels[i+1], kernel_size=2, stride=2, padding=0),
                DoubleConv(in_channels=self.decoder_channels[i], out_channels=self.decoder_channels[i+1], kernel_size=3, stride=1, padding=0)]
            ) for i in range(depth)]
        )
        
        self.conv = nn.Conv2d(in_channels=self.decoder_channels[-1], out_channels=out_channels, kernel_size=1, stride=1, padding=0)

    def _crop_feature_map(self, feature_map, x):
        '''
        Center crops feature map to the shape of x in (h, w) dimensions.
        '''
        diff_h = (feature_map.shape[-2] - x.shape[-2])/2 
        diff_w = (feature_map.shape[-1] - x.shape[-1])/2 

        assert diff_h.is_integer(), "Height difference is not a whole number."
        assert diff_w.is_integer(), "Width difference is not a whole number."

        cropped = feature_map[:, :, int(diff_h) : int(x.shape[-2] + diff_h), int(diff_w) : int(x.shape[-1] + diff_w)]

        return cropped

    def forward(self, x):
        feature_maps = dict([(index, None) for index in range(self.depth) ])

        for level in range(len(self.encoder)):
            for index in range(len(self.encoder[level])):
                x = self.encoder[level][index](x)
                if index == 0:
                    feature_maps[level] = x.clone()
                
        x = self.bottleneck(x)

        for level in range(len(self.decoder)):
            for index in range(len(self.decoder[level])):
                x = self.decoder[level][index](x)
                if index == 0:
                    feature_map_cropped = self._crop_feature_map(feature_maps[self.depth - level - 1], x)
                    x = torch.cat((feature_map_cropped, x), dim=-3)

        x = self.conv(x)

        return x

In [4]:
DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else "cpu"

In [5]:
unet = UNet(in_channels=1, out_channels=10)
unet = unet.to(DEVICE)
print(unet)

UNet(
  (encoder): ModuleList(
    (0): ModuleList(
      (0): DoubleConv(
        (conv1): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1))
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): DownSample(
        (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (1): ModuleList(
      (0): DoubleConv(
        (conv1): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1))
        (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): DownSample(
        (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (2): ModuleList(
      (0): DoubleConv(
        (conv1): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1))
        (conv2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1))
      )
      (1): DownSample(
        (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      )
    )
    (3): Mod

In [7]:
x = torch.rand((1, 1, 572, 572))
x = x.to(DEVICE)

with torch.no_grad():
    output = unet(x)

print(output.shape[-2:])

torch.Size([388, 388])
